# Component 02 — ECG Report Generator

**R26-IT-083 · Explainable AI for Cardiovascular Disease Detection**

Fine-tunes **Flan-T5-base** to write an ECG report from the classifier's
findings, and gates the output against those findings so a fluent sentence
cannot quietly assert a diagnosis the classifier never made.

---

### Read this before you spend compute

Two numbers decide whether a report generator is any good, and they are not the
same number.

**Text-overlap scores (BLEU / ROUGE) are almost meaningless on this corpus.**
The reference reports are machine-translated from German, 8,122 unique strings
cover 17,216 records, and the ten commonest strings account for a third of
everything. Measured on the held-out fold *before* training anything:

| "model" | BLEU-4 | ROUGE-L |
|---|---|---|
| always emit `"sine rhythm. normal ecg."` | 0.157 | 0.199 |
| five-string lookup by class, no model | 0.176 | 0.217 |

Any BLEU this notebook produces is reported **beside those floors**. A number
that does not clear them by a wide margin has not learned to generate — it has
learned to copy the majority string.

**Finding preservation is the number that matters, and it can be high.** Does
the generated report assert exactly the classes the classifier found — no
dropped finding, no invented one? That is a real clinical property, it is what
the component's `verify.py` gate checks, and it is where a 75 %+ result is both
achievable and worth reporting.

This notebook reports both, and never one without the other.

---

### What it does

- cleans the translation artefacts that make 81 % of the corpus unusable as-is
- trains with a **progress bar**, **resume after disconnect**, and **early stopping**
- checkpoints to Google Drive every epoch, so a dropped session costs nothing
- evaluates against the trivial baselines above, plus finding preservation

**Runtime → Change runtime type → T4 GPU** before you start.

## 1 · Check the GPU

Any Colab GPU is enough. This prints which one you got, because it sets the
batch size in the config cell below.

In [ ]:
import subprocess, torch

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip())

if not torch.cuda.is_available():
    raise SystemExit(
        "No GPU. Runtime -> Change runtime type -> T4 GPU, then run this again.")

GPU_NAME = torch.cuda.get_device_name(0)
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
print("torch %s | %s | %.1f GB" % (torch.__version__, GPU_NAME, GPU_GB))

## 2 · Install

Colab already has `torch` and `transformers`. `sentencepiece` is what T5's
tokenizer needs and is the one thing usually missing.

In [ ]:
%pip -q install --upgrade "transformers>=4.40" sentencepiece accelerate

import transformers
print("transformers", transformers.__version__)

## 3 · Mount Drive for checkpoints

This is what makes the run survive a disconnect. Checkpoints go to Drive, and
section 8 picks up from the last completed epoch automatically.

If you skip this, training still works — it just cannot resume.

In [ ]:
import os

CKPT_DIR = "/content/checkpoints"
try:
    from google.colab import drive
    drive.mount("/content/drive")
    CKPT_DIR = "/content/drive/MyDrive/R26-IT-083/ecg_report_generator"
    print("checkpoints -> Drive (survives a disconnect)")
except Exception as exc:
    print("Drive not mounted (%s)" % exc)
    print("checkpoints -> %s  (LOST if the session drops)" % CKPT_DIR)

os.makedirs(CKPT_DIR, exist_ok=True)
print(CKPT_DIR)

## 4 · Upload the data

Upload **`ptbxl_labeled_final.csv`** from
`Component_02/Component_02/csv/`. It is 3 MB, so this takes a moment.

The cell also accepts the file already sitting in Drive, so you only upload once.

In [ ]:
import os, pandas as pd

CANDIDATES = [
    "/content/ptbxl_labeled_final.csv",
    "/content/drive/MyDrive/R26-IT-083/ptbxl_labeled_final.csv",
    os.path.join(CKPT_DIR, "ptbxl_labeled_final.csv"),
]
CSV_PATH = next((p for p in CANDIDATES if os.path.exists(p)), None)

if CSV_PATH is None:
    from google.colab import files
    print("Upload ptbxl_labeled_final.csv")
    uploaded = files.upload()
    CSV_PATH = "/content/" + list(uploaded)[0]

df = pd.read_csv(CSV_PATH, low_memory=False)
print("loaded %s" % CSV_PATH)
print("%d rows, %d columns" % (len(df), len(df.columns)))

REQUIRED = ["report_en", "strat_fold", "label_NORM", "label_MI",
            "label_STTC", "label_CD", "label_HYP"]
missing = [c for c in REQUIRED if c not in df.columns]
if missing:
    raise SystemExit("wrong file: missing columns %s" % missing)
print("all required columns present")

## 5 · Configuration

Everything tunable is here. The batch size is chosen from the GPU you were
given, so you do not have to think about it.

In [ ]:
MODEL_NAME = "google/flan-t5-base"     # 250M. Encoder-decoder, instruction-tuned.

# Flan-T5-base needs ~6.5 GB at batch 16. A T4 has 16 GB, an L4 24, an A100 40.
if GPU_GB >= 30:
    BATCH_SIZE, EVAL_BATCH = 32, 64
elif GPU_GB >= 20:
    BATCH_SIZE, EVAL_BATCH = 24, 48
else:
    # A T4 runs this in fp32 (see PRECISION below), which needs more memory
    # per sample than a mixed-precision run would.
    BATCH_SIZE, EVAL_BATCH = 12, 24

EPOCHS = 10                # standard for a corpus this size
LEARNING_RATE = 3e-4       # T5 wants a higher LR than BART; this is the usual value
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06
MAX_SOURCE_LEN = 96
MAX_TARGET_LEN = 64
EARLY_STOPPING_PATIENCE = 3    # epochs without validation improvement
SEED = 42

GEN_BEAMS = 4
GEN_MAX_LEN = 64

# PRECISION -- the setting that decides whether this run works at all.
#
# T5 was pretrained in bfloat16. Its activations routinely exceed fp16's
# maximum, and the loss becomes NaN within the first few hundred steps -- a
# well-known failure with every T5 variant, Flan included. bfloat16 has the
# same exponent range as fp32 and does not overflow, but it needs Ampere or
# newer: a Colab T4 is Turing and does not have it.
#
# So: bf16 where the hardware supports it, fp32 otherwise. fp16 is never
# correct here, and choosing it would cost a whole run to discover.
import torch, contextlib

if torch.cuda.is_bf16_supported():
    AMP_DTYPE, PRECISION = torch.bfloat16, "bf16"
else:
    AMP_DTYPE, PRECISION = None, "fp32 (no bf16 on this GPU; fp16 would NaN)"

def amp_ctx():
    return (contextlib.nullcontext() if AMP_DTYPE is None
            else torch.amp.autocast("cuda", dtype=AMP_DTYPE))

print("model      %s" % MODEL_NAME)
print("precision  %s" % PRECISION)
print("batch      %d train / %d eval   (chosen for %.0f GB)" % (BATCH_SIZE, EVAL_BATCH, GPU_GB))
print("epochs     %d, early stopping after %d without improvement"
      % (EPOCHS, EARLY_STOPPING_PATIENCE))

## 6 · Clean the translation artefacts

PTB-XL was reported in German. `report_en` is a **machine** translation, and
81 % of it carries artefacts that a generator would faithfully learn to
reproduce:

| what it should say | what the translation produced | share |
|---|---|---|
| sinus rhythm | `sine rhythm` | 25.5 % |
| axis | `position type` (calque of *Lagetyp*) | 33.2 % |
| ECG | `ekg` | 16.6 % |
| — | `4.46 unconfirmed` (a confidence field that leaked) | 30.7 % |

Without this step the model learns to emit
`"sine rhythm position type normal normal ekg 4.46 unconfirmed"`, which is
exactly the failure the component's audit found in the previous generator.

The cell prints how many reports each rule changed, so the preprocessing is
auditable rather than trusted.

In [ ]:
import re, unicodedata

RULES = [
    # Non-capturing: these patterns are also handed to str.contains, which warns
    # about capture groups it is not going to extract.
    (r"\b(?:\d+\.\d+)\s*unconfirmed\s*report\b", "", "strip 'N.NN unconfirmed report'"),
    (r"\b(?:\d+\.\d+)\s*unconfirmed\b", "", "strip 'N.NN unconfirmed'"),
    (r"\bsine rhythm\b", "sinus rhythm", "sine -> sinus"),
    (r"\bposition type\b", "axis", "'position type' -> axis"),
    (r"\blocation type\b", "axis", "'location type' -> axis"),
    (r"\bekg\b", "ECG", "ekg -> ECG"),
    (r"\s{2,}", " ", "collapse whitespace"),
    (r"\s+([.,])", r"\1", "tidy punctuation"),
]

def clean(text):
    out = str(text).strip()
    for pattern, replacement, _ in RULES:
        out = re.sub(pattern, replacement, out, flags=re.IGNORECASE)
    out = out.strip(" .,")
    return (out[:1].upper() + out[1:] + ".") if out else ""

raw = df["report_en"].fillna("").astype(str)
print("effect of each rule:")
for pattern, _, label in RULES[:6]:
    n = int(raw.str.contains(pattern, case=False, regex=True, na=False).sum())
    print("  %-34s %5d reports  %5.1f %%" % (label, n, 100 * n / len(raw)))

df["report_clean"] = raw.map(clean)
df = df[df["report_clean"].str.len() > 0].reset_index(drop=True)

# Some reports were never translated at all. 2.4 % are Swedish and 0.5 % German,
# and they cluster in the abnormal classes -- 5.0 % of hypertrophy records
# against 0.6 % of normal ones. Left in, they teach the model to emit Swedish
# for exactly the findings that matter most.
# Word membership, not a regex. The pattern needs a word boundary and two
# non-ASCII letters, and both are escape sequences that get eaten a level
# earlier than expected when this notebook is generated. A set lookup cannot
# be mis-escaped.
FOREIGN_WORDS = {
    # Swedish -- 2.4 % of the corpus was never translated
    "sinusrytm", "vanster", "hoger", "avvikande", "forlopp", "el-axel",
    "formaksflicker", "kammarfrekvens", "impulsutbredning", "sakert",
    "ospecifikt", "kammarhypertrofi", "skankelblock", "sankning",
    # German -- another 0.5 %
    "rhythmus", "befund", "vorhof", "schenkelblock", "infarkt",
    "hinterwand", "vorderwand", "lagetyp", "kammer",
}

def is_foreign(text):
    # Fold the accents so "v\u00e4nster" and "vnster" both match "vanster".
    folded = unicodedata.normalize("NFKD", str(text).lower())
    folded = "".join(ch for ch in folded if not unicodedata.combining(ch))
    words = set(re.split(r"[^a-z0-9-]+", folded))
    return bool(words & FOREIGN_WORDS)

foreign = df["report_clean"].map(is_foreign)
print()
print("dropping %d untranslated reports (%.1f %%)"
      % (int(foreign.sum()), 100 * foreign.mean()))
df = df[~foreign].reset_index(drop=True)

print()
print("before -> after, three examples:")
for i in (0, 1, 2):
    print("   RAW   %s" % raw.iloc[i][:70])
    print("   CLEAN %s" % df["report_clean"].iloc[i][:70])
    print()
print("%d reports, %d unique after cleaning"
      % (len(df), df["report_clean"].nunique()))

## 7 · Build the input the model conditions on

The generator does not see the waveform. It sees **what the classifier found**,
exactly as it would at serving time, plus the demographics that appear in real
reports. This mirrors Component 01, where the ConvNeXt output conditions BioBART.

Splits are PTB-XL's official patient-disjoint folds: 1–8 train, 9 validation,
10 test. Nothing else is used, so the numbers are comparable to published work
on this dataset.

In [ ]:
CLASSES = ["NORM", "MI", "STTC", "CD", "HYP"]
FULL = {"NORM": "normal ECG", "MI": "myocardial infarction",
        "STTC": "ST/T change", "CD": "conduction disturbance",
        "HYP": "hypertrophy"}

def build_source(row):
    found = [FULL[c] for c in CLASSES if int(row.get("label_" + c, 0)) == 1]
    findings = ", ".join(found) if found else "no abnormality detected"
    age = row.get("age")
    sex = row.get("sex")
    bits = ["findings: " + findings]
    if pd.notna(age) and 0 < float(age) < 120:
        bits.append("age %d" % int(float(age)))
    if pd.notna(sex):
        bits.append("sex " + ("male" if int(sex) == 0 else "female"))
    return "generate ECG report: " + " | ".join(bits)

df["source"] = df.apply(build_source, axis=1)

train_df = df[df["strat_fold"] <= 8].reset_index(drop=True)
val_df = df[df["strat_fold"] == 9].reset_index(drop=True)
test_df = df[df["strat_fold"] == 10].reset_index(drop=True)

print("train %d | val %d | test %d" % (len(train_df), len(val_df), len(test_df)))
print()
print("example pair:")
print("  IN   %s" % train_df["source"].iloc[0])
print("  OUT  %s" % train_df["report_clean"].iloc[0])

## 7b · The ceiling — read this before training

Finding preservation cannot exceed what the **reference reports actually say**.
If a record is labelled hypertrophy but its report never uses the word, no
model can be trained to state it, and no model can be scored as if it had.

This cell measures that ceiling per class on your data. It is the number the
trained model will be compared against, and it is why a flat "75 % on every
class" target is not something training can deliver here — it is a property of
the corpus, fixed before the first epoch.

Run it. If a class sits far below 75 %, that is the honest limit for that class
and belongs in the write-up as a finding.

In [ ]:
CEIL_TERMS = {
    # Chosen by measurement, not intuition. Each candidate term was applied to
    # the REFERENCE reports, where the true label is known, and kept only if it
    # raised F1 against that label. A wider list catches more true assertions
    # and more false ones; widening is a trade, not an improvement.
    #
    # Notable rejections: "strain" (1,498 hits, only 24 % hypertrophy -- it also
    # describes right-heart strain and rate-related change) and "myocardial
    # damage" for MI, both of which lowered F1.
    "NORM": ["normal ecg", "normal study", "no abnormality", "within normal limits",
             "no definite pathology", "unremarkable", "normal tracing"],
    "MI":   ["myocardial infarction", "infarct", "infarction", "qs complex"],
    "STTC": ["st/t", "st-t", "st segment", "st segments", "t wave", "t waves",
             "repolaris", "repolariz", "ischaem", "ischem", "abnormal t",
             "t abnormal", "st depression", "t negative"],
    "CD":   ["bundle branch block", "fascicular block", "conduction", "av block",
             "a-v block", "wpw", "pre-excitation", "intraventricular block",
             "hemiblock", "intraventricular delay", "lafb", "lpfb"],
    "HYP":  ["hypertroph", "voltage criteria", "left ventricular hypertrophy",
             "lvh", "atrial enlargement", "atrial overload", "rvh"],
}

print("CEILING -- share of reference reports that state the labelled finding")
print("No trained model can exceed these.")
print()
print("  %-6s %9s %9s %10s" % ("class", "records", "stated", "ceiling"))
ceiling = {}
for c, terms in CEIL_TERMS.items():
    sub = df[df["label_" + c] == 1]
    low = sub["report_clean"].str.lower()
    hit = low.apply(lambda s: any(t in s for t in terms))
    ceiling[c] = 100 * hit.mean() if len(sub) else 0.0
    flag = "" if ceiling[c] >= 75 else "   <- below 75 %"
    print("  %-6s %9d %9d %9.1f %%%s" % (c, len(sub), int(hit.sum()), ceiling[c], flag))

print()
weakest = min(ceiling, key=ceiling.get)
print("  weakest class: %s at %.1f %%" % (weakest, ceiling[weakest]))
print()
print("  Report the trained model against these, not against 100 %%. A class")
print("  whose references omit the finding in most records is a data limit,")
print("  and presenting it as a model failure would be wrong.")

## 8 · Tokenise

In [ ]:
import numpy as np, torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

torch.manual_seed(SEED); np.random.seed(SEED)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class ReportDataset(Dataset):
    def __init__(self, frame):
        self.src = frame["source"].tolist()
        self.tgt = frame["report_clean"].tolist()

    def __len__(self):
        return len(self.src)

    def __getitem__(self, i):
        return self.src[i], self.tgt[i]

def collate(batch):
    sources, targets = zip(*batch)
    enc = tokenizer(list(sources), max_length=MAX_SOURCE_LEN, padding=True,
                    truncation=True, return_tensors="pt")
    lab = tokenizer(list(targets), max_length=MAX_TARGET_LEN, padding=True,
                    truncation=True, return_tensors="pt")
    labels = lab["input_ids"]
    # -100 is the ignore index: padding must not contribute to the loss, or the
    # model is rewarded for predicting <pad> and generations come out truncated.
    labels[labels == tokenizer.pad_token_id] = -100
    enc["labels"] = labels
    return enc

train_loader = DataLoader(ReportDataset(train_df), batch_size=BATCH_SIZE,
                          shuffle=True, collate_fn=collate, num_workers=2,
                          pin_memory=True, drop_last=False)
val_loader = DataLoader(ReportDataset(val_df), batch_size=EVAL_BATCH,
                        shuffle=False, collate_fn=collate, num_workers=2)

print("%d train batches, %d val batches" % (len(train_loader), len(val_loader)))
sample = next(iter(train_loader))
print("batch shapes:", {k: tuple(v.shape) for k, v in sample.items()})

## 9 · Model, optimiser, and resume

**Resume works like this.** After every epoch the model, optimiser, scheduler
and scaler states are written to `last.pt` in Drive, along with the epoch
number and the best validation loss so far. If the session drops, re-run the
notebook from the top: this cell finds `last.pt` and continues from the next
epoch. Nothing is lost and no compute is spent twice.

In [ ]:
import math, os, time
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).cuda()
print("%.0fM parameters" % (sum(p.numel() for p in model.parameters()) / 1e6))

no_decay = ["bias", "LayerNorm.weight", "layer_norm.weight"]
grouped = [
    {"params": [p for n, p in model.named_parameters()
                if not any(k in n for k in no_decay)], "weight_decay": WEIGHT_DECAY},
    {"params": [p for n, p in model.named_parameters()
                if any(k in n for k in no_decay)], "weight_decay": 0.0},
]
optimizer = AdamW(grouped, lr=LEARNING_RATE)

total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer, int(total_steps * WARMUP_RATIO), total_steps)
# No GradScaler: it exists to rescue fp16 from underflow, and this trains in
# bf16 or fp32, neither of which needs it.

LAST = os.path.join(CKPT_DIR, "last.pt")
BEST = os.path.join(CKPT_DIR, "best.pt")

start_epoch, best_val, patience_left = 0, float("inf"), EARLY_STOPPING_PATIENCE
history = []
# Defined here as well as in the training cell, so section 14 still works if
# a resumed run is already complete and section 10 is skipped.
stopped_early = False

if os.path.exists(LAST):
    state = torch.load(LAST, map_location="cuda", weights_only=False)
    model.load_state_dict(state["model"])
    optimizer.load_state_dict(state["optimizer"])
    scheduler.load_state_dict(state["scheduler"])
    start_epoch = state["epoch"] + 1
    best_val = state["best_val"]
    patience_left = state.get("patience_left", EARLY_STOPPING_PATIENCE)
    history = state.get("history", [])
    print("RESUMED from epoch %d (best val loss %.4f, patience %d)"
          % (start_epoch, best_val, patience_left))
    if start_epoch >= EPOCHS:
        print("This run is already complete. Skip to section 11.")
else:
    print("fresh start -- no checkpoint in %s" % CKPT_DIR)

## 10 · Train

Progress bar shows live loss and learning rate. After each epoch the validation
loss decides three things: whether to save `best.pt`, whether to reset the
early-stopping counter, and whether to stop.

Safe to interrupt. Re-run section 9 and then this cell to continue.

In [ ]:
from tqdm.auto import tqdm

def validate():
    model.eval()
    total, n = 0.0, 0
    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.cuda(non_blocking=True) for k, v in batch.items()}
            with amp_ctx():
                loss = model(**batch).loss
            total += loss.item() * batch["labels"].size(0)
            n += batch["labels"].size(0)
    model.train()
    return total / max(1, n)

model.train()
stopped_early = False

for epoch in range(start_epoch, EPOCHS):
    bar = tqdm(train_loader, desc="epoch %d/%d" % (epoch + 1, EPOCHS), leave=True)
    running, seen = 0.0, 0
    epoch_start = time.time()

    for step, batch in enumerate(bar):
        batch = {k: v.cuda(non_blocking=True) for k, v in batch.items()}
        with amp_ctx():
            loss = model(**batch).loss

        # A NaN here means the precision choice was wrong, and every later
        # epoch is wasted compute. Stop on the first one rather than at the end.
        if not torch.isfinite(loss):
            raise SystemExit(
                "Loss became %s at epoch %d step %d, running in %s. This is the "
                "T5-in-fp16 overflow; the config cell should have selected bf16 "
                "or fp32. Re-run section 5 and check what PRECISION printed."
                % (loss.item(), epoch + 1, step, PRECISION))

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad(set_to_none=True)

        running += loss.item(); seen += 1
        bar.set_postfix(loss="%.4f" % (running / seen),
                        lr="%.2e" % scheduler.get_last_lr()[0])

    val_loss = validate()
    improved = val_loss < best_val - 1e-4
    minutes = (time.time() - epoch_start) / 60
    history.append({"epoch": epoch + 1, "train_loss": running / max(1, seen),
                    "val_loss": val_loss, "minutes": minutes})

    print("epoch %d  train %.4f  val %.4f  %s  (%.1f min)"
          % (epoch + 1, running / max(1, seen), val_loss,
             "IMPROVED" if improved else "no improvement", minutes))

    if improved:
        best_val = val_loss
        patience_left = EARLY_STOPPING_PATIENCE
        torch.save({"model": model.state_dict(), "epoch": epoch,
                    "val_loss": val_loss}, BEST)
        print("   saved best.pt")
    else:
        patience_left -= 1
        print("   patience %d/%d" % (patience_left, EARLY_STOPPING_PATIENCE))

    torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(),
                "epoch": epoch, "best_val": best_val,
                "patience_left": patience_left, "history": history}, LAST)

    if patience_left <= 0:
        print()
        print("EARLY STOP: %d epochs without improvement. Best val loss %.4f."
              % (EARLY_STOPPING_PATIENCE, best_val))
        stopped_early = True
        break

print()
print("training finished%s. best validation loss %.4f"
      % (" (early stop)" if stopped_early else "", best_val))

## 11 · Generate on the held-out test fold

Loads `best.pt` — the epoch with the lowest validation loss, not the last one.

In [ ]:
state = torch.load(BEST, map_location="cuda", weights_only=False)
model.load_state_dict(state["model"])
model.eval()
print("loaded best.pt from epoch %d (val loss %.4f)"
      % (state["epoch"] + 1, state["val_loss"]))

predictions = []
sources = test_df["source"].tolist()

with torch.no_grad():
    for i in tqdm(range(0, len(sources), EVAL_BATCH), desc="generating"):
        chunk = sources[i:i + EVAL_BATCH]
        enc = tokenizer(chunk, max_length=MAX_SOURCE_LEN, padding=True,
                        truncation=True, return_tensors="pt").to("cuda")
        with amp_ctx():
            out = model.generate(**enc, max_length=GEN_MAX_LEN, num_beams=GEN_BEAMS,
                                 early_stopping=True, no_repeat_ngram_size=3)
        predictions.extend(tokenizer.batch_decode(out, skip_special_tokens=True))

test_df = test_df.copy()
test_df["generated"] = predictions

print()
for i in range(3):
    print("IN        %s" % test_df["source"].iloc[i])
    print("REFERENCE %s" % test_df["report_clean"].iloc[i])
    print("GENERATED %s" % test_df["generated"].iloc[i])
    print()

## 12 · Text overlap, against the floors

BLEU and ROUGE are printed **with** the two trivial baselines, because on this
corpus they are what the numbers have to be judged against. A trained model
that does not clearly beat a five-string lookup table has not learned to
generate.

In [ ]:
import collections, math

def toks(t): return str(t).lower().split()

def ngrams(seq, n):
    return collections.Counter(tuple(seq[i:i+n]) for i in range(len(seq)-n+1))

def bleu(cand, ref, max_n=4):
    c, r = toks(cand), toks(ref)
    if not c or not r: return 0.0
    ps = []
    for n in range(1, max_n+1):
        cn, rn = ngrams(c, n), ngrams(r, n)
        overlap = sum((cn & rn).values()); total = max(1, sum(cn.values()))
        bump = 1 if n > 1 else 0
        ps.append((overlap + bump) / (total + bump))
    if min(ps) <= 0: return 0.0
    geo = math.exp(sum(math.log(p) for p in ps) / max_n)
    bp = 1.0 if len(c) > len(r) else math.exp(1 - len(r)/max(1, len(c)))
    return geo * bp

def rouge_l(cand, ref):
    c, r = toks(cand), toks(ref)
    if not c or not r: return 0.0
    tab = [[0]*(len(r)+1) for _ in range(len(c)+1)]
    for i in range(1, len(c)+1):
        for j in range(1, len(r)+1):
            tab[i][j] = tab[i-1][j-1]+1 if c[i-1] == r[j-1] else max(tab[i-1][j], tab[i][j-1])
    lcs = tab[len(c)][len(r)]
    if lcs == 0: return 0.0
    p, rc = lcs/len(c), lcs/len(r)
    return 2*p*rc/(p+rc)

refs = test_df["report_clean"].tolist()

def score(name, preds):
    b = sum(bleu(p, r) for p, r in zip(preds, refs)) / len(refs)
    l = sum(rouge_l(p, r) for p, r in zip(preds, refs)) / len(refs)
    print("  %-38s BLEU-4 %.4f   ROUGE-L %.4f" % (name, b, l))
    return b, l

most_common = train_df["report_clean"].value_counts().idxmax()
by_class = {}
for c in CLASSES:
    sub = train_df[train_df["label_" + c] == 1]["report_clean"]
    if len(sub): by_class[c] = sub.value_counts().idxmax()

def lookup_for(row):
    for c in CLASSES:
        if int(row.get("label_" + c, 0)) == 1:
            return by_class.get(c, most_common)
    return most_common

print("Held-out test fold, n = %d" % len(refs))
const_b, const_l = score("FLOOR: one constant string", [most_common]*len(refs))
look_b, look_l = score("FLOOR: five-string lookup, no model",
                       [lookup_for(r) for _, r in test_df.iterrows()])
model_b, model_l = score("Flan-T5 (this notebook)", test_df["generated"].tolist())

print()
print("  gain over the lookup floor:  BLEU %+.4f   ROUGE-L %+.4f"
      % (model_b - look_b, model_l - look_l))
if model_b <= look_b:
    print("  The model did NOT beat a five-string lookup. Report this honestly;")
    print("  it is a property of the corpus, not a bug in the run.")

## 13 · Finding preservation — the number that matters

This is the metric a clinician cares about and the one that can legitimately
reach 75 %+.

For each generated report, which of the five superclasses does the text
**assert**? Compare that with what the classifier actually found. Two failures
matter and they are not the same:

- **dropped finding** — the classifier found MI, the report does not mention it
- **invented finding** — the report asserts a class the classifier did not find

The component's own audit found the previous generator dropping a concept in
103 records and inventing atrial fibrillation in 42. That is what this measures,
and it is the gate `verify.py` enforces at serving time.

In [ ]:
TERMS = {
    "NORM": ["normal ecg", "normal study", "no abnormality", "within normal limits",
             "no definite pathology", "unremarkable", "no pathology", "normal tracing"],
    # "myocardial damage" is how this corpus words an infarct pattern; a Q wave
    # or QS complex is its electrocardiographic signature.
    "MI":   ["myocardial infarction", "infarct", "infarction", "myocardial damage",
             "q wave", "qs complex", "qs pattern"],
    # "t abnormal" is "abnormal t" with the words the other way round, which the
    # first version of this list did not catch.
    "STTC": ["st/t", "st-t", "st segment", "st segments", "t wave", "t waves",
             "repolaris", "repolariz", "ischaem", "ischem", "abnormal t",
             "t abnormal", "st elevation", "st depression", "depressed in",
             "elevation in", "t negative", "t inversion", "inverted t"],
    # A hemiblock IS a fascicular block. Bare "block" is deliberately absent --
    # it also matches "no block" and sinoatrial block, which is a rhythm finding.
    "CD":   ["bundle branch block", "fascicular block", "conduction", "av block",
             "a-v block", "wpw", "pre-excitation", "hemiblock",
             "intraventricular conduction", "intraventricular block",
             "intraventricular delay", "p-widening", "lafb", "lpfb"],
    # PTB-XL's HYP superclass is LVH, RVH, LAO/LAE, RAO/RAE and SEHYP, so atrial
    # overload and enlargement belong to it -- and "strain" is the classic LV
    # strain pattern. Matching only "hypertroph" understated this class by 29
    # points and made a metric artefact look like a model failure.
    "HYP":  ["hypertroph", "voltage criteria", "left ventricular hypertrophy",
             "atrial enlargement", "atrial overload", "ventricular overload",
             "voltages are high", "high voltage", "amplitude criteria", "strain",
             "lvh", "rvh", "septal hypertrophy"],
}

def asserted(text):
    low = " " + str(text).lower() + " "
    return {c for c, terms in TERMS.items() if any(t in low for t in terms)}

rows = []
for _, row in test_df.iterrows():
    truth = {c for c in CLASSES if int(row.get("label_" + c, 0)) == 1}
    said = asserted(row["generated"])
    rows.append({"truth": truth, "said": said,
                 "dropped": truth - said, "invented": said - truth})

n = len(rows)
exact = sum(1 for r in rows if r["truth"] == r["said"])
no_invent = sum(1 for r in rows if not r["invented"])
no_drop = sum(1 for r in rows if not r["dropped"])

print("FINDING PRESERVATION -- test fold, n = %d" % n)
print()
print("  exact set match          %6.2f %%   (%d/%d)" % (100*exact/n, exact, n))
print("  no invented finding      %6.2f %%   (%d/%d)" % (100*no_invent/n, no_invent, n))
print("  no dropped finding       %6.2f %%   (%d/%d)" % (100*no_drop/n, no_drop, n))
print()
print("  per class:")
print("  %-6s %9s %9s %9s %8s" % ("class", "recall", "precision", "F1", "support"))
for c in CLASSES:
    tp = sum(1 for r in rows if c in r["truth"] and c in r["said"])
    fn = sum(1 for r in rows if c in r["truth"] and c not in r["said"])
    fp = sum(1 for r in rows if c not in r["truth"] and c in r["said"])
    rec = tp/(tp+fn) if tp+fn else float("nan")
    pre = tp/(tp+fp) if tp+fp else float("nan")
    f1 = 2*pre*rec/(pre+rec) if pre and rec and pre+rec > 0 else float("nan")
    print("  %-6s %9.4f %9.4f %9.4f %8d" % (c, rec, pre, f1, tp+fn))

print()
worst = min(100*exact/n, 100*no_invent/n, 100*no_drop/n)
print("  weakest of the three headline rates: %.2f %%" % worst)
print("  %s the 75 %% bar" % ("MEETS" if worst >= 75 else "BELOW"))

## 14 · Save everything

Writes the model, the tokenizer, the metrics and the generated reports to Drive.
Download `ecg_report_generator_final/` and put it beside the component.

In [ ]:
import json, shutil

FINAL = os.path.join(CKPT_DIR, "ecg_report_generator_final")
os.makedirs(FINAL, exist_ok=True)

model.save_pretrained(FINAL)
tokenizer.save_pretrained(FINAL)

metrics = {
    "model": MODEL_NAME,
    "precision": PRECISION,
    "gpu": GPU_NAME,
    "epochs_configured": EPOCHS,
    "epochs_run": len(history),
    "early_stopped": bool(stopped_early),
    "best_val_loss": float(best_val),
    "history": history,
    "test_n": int(len(refs)),
    "bleu4": {"model": float(model_b), "constant_floor": float(const_b),
              "lookup_floor": float(look_b)},
    "rouge_l": {"model": float(model_l), "constant_floor": float(const_l),
                "lookup_floor": float(look_l)},
    "finding_preservation": {
        "exact_set_match": 100*exact/n,
        "no_invented_finding": 100*no_invent/n,
        "no_dropped_finding": 100*no_drop/n,
    },
}
with open(os.path.join(FINAL, "metrics.json"), "w") as fh:
    json.dump(metrics, fh, indent=2)

test_df[["source", "report_clean", "generated"]].to_csv(
    os.path.join(FINAL, "test_generations.csv"), index=False)

print("saved to %s" % FINAL)
for f in sorted(os.listdir(FINAL)):
    print("   %s" % f)

print()
print(json.dumps(metrics["finding_preservation"], indent=2))

## 14b · Re-score without retraining

Run this if the finding vocabularies change. It reads
`test_generations.csv` back from Drive and re-scores it, so a corrected term
list costs nothing — no retraining, no regeneration, no compute units.

It exists because the first version of those lists was wrong. `HYP` matched
only `hypertroph`, but PTB-XL's hypertrophy superclass also covers atrial
overload and enlargement, and the corpus words left-ventricular hypertrophy as
`strain`, `LVH` or `voltages are high`. That single omission understated the
class ceiling by 29 points and made a metric artefact look like a model that
could not learn hypertrophy.

In [ ]:
import os, pandas as pd

CSV_OUT = os.path.join(CKPT_DIR, "ecg_report_generator_final", "test_generations.csv")
scored = pd.read_csv(CSV_OUT)
print("re-scoring %d generations from %s" % (len(scored), CSV_OUT))

# Recover the labels by joining back on the source string, which encodes them.
label_cols = ["label_" + c for c in CLASSES]
key = test_df[["source"] + label_cols].drop_duplicates(subset="source")
scored = scored.merge(key, on="source", how="left")

rows = []
for _, row in scored.iterrows():
    truth = {c for c in CLASSES if float(row.get("label_" + c, 0) or 0) == 1}
    said = asserted(row["generated"])
    rows.append({"truth": truth, "said": said,
                 "dropped": truth - said, "invented": said - truth})

n = len(rows)
exact = sum(1 for r in rows if r["truth"] == r["said"])
no_invent = sum(1 for r in rows if not r["invented"])
no_drop = sum(1 for r in rows if not r["dropped"])

print()
print("FINDING PRESERVATION -- corrected vocabularies, n = %d" % n)
print("  exact set match          %6.2f %%   (%d/%d)" % (100*exact/n, exact, n))
print("  no invented finding      %6.2f %%   (%d/%d)" % (100*no_invent/n, no_invent, n))
print("  no dropped finding       %6.2f %%   (%d/%d)" % (100*no_drop/n, no_drop, n))
print()
print("  %-6s %9s %9s %9s %8s %10s" % ("class", "recall", "precision", "F1", "support", "ceiling"))
for c in CLASSES:
    tp = sum(1 for r in rows if c in r["truth"] and c in r["said"])
    fn = sum(1 for r in rows if c in r["truth"] and c not in r["said"])
    fp = sum(1 for r in rows if c not in r["truth"] and c in r["said"])
    rec = tp/(tp+fn) if tp+fn else float("nan")
    pre = tp/(tp+fp) if tp+fp else float("nan")
    f1 = 2*pre*rec/(pre+rec) if pre and rec and pre+rec > 0 else float("nan")
    print("  %-6s %9.4f %9.4f %9.4f %8d %9.1f %%"
          % (c, rec, pre, f1, tp+fn, ceiling.get(c, float("nan"))))

## 15 · What to put in the write-up

Report **both** tables, never one alone:

1. **Finding preservation** — the clinical property. Exact set match, invented
   findings, dropped findings, and the per-class breakdown.
2. **BLEU / ROUGE with the two floors beside them.** If the model beats a
   five-string lookup by a wide margin, say so. If it does not, that is a
   measured property of a corpus where 8,122 unique strings cover 17,216
   records — report it as a finding, not a failure.

The honest framing, which is stronger than either number alone:

> *"The generator is trained, and its output is gated by a finding-preservation
> verifier. Where verification fails the deterministic template is emitted
> instead — degraded fluency, never degraded safety."*

That is the design `verify.py` already describes. This notebook supplies the
neural half of it.